# 32 — Observability, Monitoring & Logging

**Role:** Senior AWS DevOps Engineer | **Focus:** CloudWatch, Grafana, Prometheus, Splunk, Datadog, ELK

---

## Section A: Observability Foundations

### Q1: Three Pillars of Observability
**Question:** What are the three pillars of observability? How do they differ and complement each other?

**Expected Answer:**
- **Metrics**: Numeric time-series data (CPU, latency, error rate). Cheap to store, good for alerting. Tool: Prometheus + Grafana.
- **Logs**: Structured event records. Rich context but expensive at scale. Tool: ELK, Splunk, CloudWatch Logs.
- **Traces**: Request flow across services. Essential for microservices debugging. Tool: OpenTelemetry, X-Ray, Jaeger.
- **Correlation**: Trace ID links a log entry to a metric spike to the exact distributed call chain.

---

### Q2: Prometheus Architecture in EKS
**Question:** How do you deploy Prometheus in an EKS cluster? How does service discovery work?

**Expected Answer:**
- **Deployment**: Prometheus Operator via Helm (kube-prometheus-stack). Creates Prometheus, Alertmanager, Grafana.
- **Service Discovery**: `ServiceMonitor` and `PodMonitor` CRDs auto-discover targets by labels.
- **Storage**: Local PVC for short-term. Thanos or Cortex for long-term, multi-cluster storage.
- **AWS Managed**: Amazon Managed Prometheus (AMP) — fully managed, compatible with PromQL.
- **Scrape config**: Pods expose `/metrics` endpoint; Prometheus scrapes on interval.

---

### Q3: Grafana Dashboard Design
**Question:** What dashboards do you set up for a production EKS platform on day one?

**Expected Answer:**
- **Cluster Overview**: Node count, CPU/memory utilization, pod counts by status.
- **Namespace Dashboard**: Per-namespace resource usage vs. quotas.
- **Application RED**: Rate (requests/sec), Errors (5xx %), Duration (p50/p95/p99 latency).
- **Node Health**: Disk pressure, memory pressure, PID pressure, network I/O.
- **Cost Dashboard**: Kubecost integration — cost per namespace/team.
- **Alerts**: Linked panels → clicking a spike opens the relevant log search.

## Section B: Centralized Logging

### Q4: Log Architecture for EKS
**Question:** How do you collect logs from EKS pods and send them to a centralized logging system?

**Expected Answer:**
- **DaemonSet approach**: Fluent Bit (lightweight) or Fluentd as a DaemonSet on every node.
- **Sidecar approach**: Per-pod log forwarder. More isolated but higher resource overhead.
- **Pipeline**: Pod stdout → Node `/var/log/containers` → Fluent Bit → Elasticsearch/Splunk/CloudWatch.
- **Structured logging**: Apps must log in JSON. Fields: `timestamp`, `level`, `message`, `trace_id`, `service`.
- **Retention**: Hot (7 days searchable), Warm (30 days), Cold (S3 archive for compliance).

---

### Q5: ELK vs. Splunk vs. Datadog
**Question:** You need to choose a centralized logging platform. Compare ELK, Splunk, and Datadog.

**Expected Answer:**

| Criteria | ELK (Self-managed) | Splunk | Datadog |
|---|---|---|---|
| Cost | Open-source + infra | Expensive per GB | Per-host pricing |
| Ops Overhead | High (manage ES cluster) | Medium (Splunk Cloud) | Low (SaaS) |
| Search Power | KQL, Lucene | SPL (very powerful) | Built-in faceted search |
| Integration | Manual setup | Enterprise-grade | Best for cloud-native |
| Best For | Budget-conscious teams | Enterprise compliance | Full observability suite |

- **Decision factors**: Budget, team size, compliance needs, existing tooling.

---

### Q6: CloudWatch Deep Dive
**Question:** When do you use CloudWatch vs. Prometheus/Grafana? Can they coexist?

**Expected Answer:**
- **CloudWatch**: Best for AWS-native services (RDS, Lambda, ALB). Built-in, no setup.
- **Prometheus**: Best for Kubernetes workloads, custom application metrics.
- **Coexistence**: Use CloudWatch for infra metrics (RDS CPU, Lambda invocations) and Prometheus for app metrics. Grafana can query both as data sources.
- **CloudWatch Container Insights**: EKS metrics in CloudWatch without Prometheus. Good starting point.
- **Cost trap**: CloudWatch custom metrics and log ingestion can get expensive at scale.

## Section C: Alerting & On-Call

### Q7: Alerting Strategy
**Question:** How do you design alerts that are actionable and avoid alert fatigue?

**Expected Answer:**
- **Alert on symptoms, not causes**: Alert on "error rate > 5%" not "CPU > 80%".
- **Severity levels**: P1 (pages on-call), P2 (Slack notification), P3 (ticket auto-created).
- **Every alert needs a runbook**: Link in the alert annotation.
- **SLO-based alerting**: Burn rate alerts ("At this error rate, we'll breach our 99.9% SLO in 6 hours").
- **Deduplication**: Use Alertmanager grouping and inhibition rules.
- **Review regularly**: If an alert fires and no one acts on it, delete or downgrade it.

---

### Q8: Distributed Tracing
**Question:** A user reports slow page load. The request hits API Gateway → ALB → EKS (3 microservices) → RDS. How do you find the bottleneck?

**Expected Answer:**
- **Trace ID propagation**: OpenTelemetry SDK in each service, auto-injects `traceparent` header.
- **Collector**: OTel Collector as a DaemonSet → exports to X-Ray, Jaeger, or Datadog.
- **Waterfall view**: See time spent in each service. Identify the slow span.
- **Correlate with logs**: Filter logs by trace ID to see error messages in the slow service.
- **RDS**: Enable Performance Insights to correlate slow queries with trace timestamps.

---

### Q9: SLOs, SLIs, and Error Budgets
**Question:** Define SLO, SLI, and Error Budget. How do you use them in practice?

**Expected Answer:**
- **SLI (Indicator)**: The metric. e.g., "Percentage of requests completing in < 200ms."
- **SLO (Objective)**: The target. e.g., "99.9% of requests complete in < 200ms over 30 days."
- **Error Budget**: `100% - SLO = 0.1%` allowable failures. If budget is exhausted, freeze deployments and focus on reliability.
- **Implementation**: Prometheus recording rules calculate SLI, Grafana dashboard shows budget burn rate, Alertmanager fires when burn rate is high.